In [1]:
!pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Web scrapping de France travail

In [2]:
from dotenv import load_dotenv
import os
load_dotenv
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")

In [3]:
import requests

In [4]:
URL = "https://entreprise.francetravail.fr/connexion/oauth2/access_token?realm=/partenaire"

def get_token() :
    response = requests.post(URL, data = {
        "grant_type" : "client_credentials",
        "client_id" : client_id,
        "client_secret" : client_secret,
        "scope" : "api_offresdemploiv2 o2dsoffre"
    }, headers = {"Content-Type": "application/x-www-form-urlencoded"})

    response.raise_for_status()
    token = response.json()["access_token"]
    print(f" Début du token: {token[:40]}")
    return token


TOKEN = get_token()


 Début du token: 8HXiXUpT2ksHx0tfDw9Wd0Tz_Gw


In [5]:
API_URL = "https://api.francetravail.io/partenaire/offresdemploi/v2/offres/search"

def fetch_offres(token, mots_cles, start = 0, end = 4):
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept" : "application/json"
    }

    params = {
        "motsCles": mots_cles,
        "range": f"{start}-{end}"
    }

    r = requests.get(API_URL, headers = headers, params = params)
    r.raise_for_status()
    return r.json()

test = fetch_offres(TOKEN, "data analyste")
offres = test.get("resultats", [])
print(f"Nombre d'offres reçues: {len(offres)}")
for o in offres:
    print(f" -{o["intitule"]} | {o.get('lieuTravail', {}).get('libelle', '?')} | {o.get('salaire', {}).get('libelle', 'salaire non renseigné')}")

Nombre d'offres reçues: 5
 -ANALYSTE DATA          (H/F) | 10 - NOGENT SUR SEINE | salaire non renseigné
 -Analyste Data - Plateformes de Digital Learning (H/F) | 92 - Courbevoie | Mensuel de 1407.0 Euros à 1638.0 Euros sur 12.0 mois
 -Chargé d'Analyse et Master Data Achats (F/H) | 93 - Saint-Ouen-sur-Seine | Annuel de 40000.0 Euros à 45000.0 Euros sur 12.0 mois
 -Master Data Analyste Industriel (H/F) | 75 - PARIS 09 | salaire non renseigné
 -Analyste informatique KELIA (H/F) | 75 - Paris 13e Arrondissement | Mensuel de 2000.0 Euros à 6000.0 Euros sur 12.0 mois


In [6]:
import json, time, os
from tqdm import tqdm
from datetime import datetime

mots_cles = ["data analyst", "data engineer", "data scientist", "machine learning", "analyste données", "analyste quantitatif"]

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok = True)

toutes_offres = []

TOKEN = get_token()

for mot_cle in mots_cles :
    print(f"\nRécupération : '{mot_cle}'")
    offres_recherchees = []
    start = 0
    page = 1

    while True:
        end = start + 149

        try: 
            result = fetch_offres(TOKEN, mot_cle, start = start, end = end)
            offres = result.get("resultats", [])

            if not offres:
                print(f"Page {page}: vide, arrêt.")
                break

            for o in offres :
                o["_type_de_poste"] = mot_cle
                o["_heure_de_collecte"] = datetime.now().isoformat()

            offres_recherchees.extend(offres)
            print(f"Page {page} : +{len(offres)} offres (total'{mot_cle}' : {len(offres_recherchees)})")

            if len(offres) < 150:
                print(f"Dernière page atteinte")
                break

            start += 150
            page += 1
            time.sleep(1)

        except requests.HTTPError as e:
            print(f" Erreur HTTP page {page} : {e}")
            break
        
    print(f" Total pour '{mot_cle}': {len(offres_recherchees)} offres")
    toutes_offres.extend(offres_recherchees)

print(f"\n{'='*45}")
print(f"Total Général: {len(toutes_offres)} offres collectées")
print(f"{'='*45}")

 Début du token: YvnfNo8PjfoOlwgAwwE6EruvF_4

Récupération : 'data analyst'
Page 1 : +150 offres (total'data analyst' : 150)
Page 2 : +150 offres (total'data analyst' : 300)
Page 3 : +54 offres (total'data analyst' : 354)
Dernière page atteinte
 Total pour 'data analyst': 354 offres

Récupération : 'data engineer'
Page 1 : +150 offres (total'data engineer' : 150)
Page 2 : +150 offres (total'data engineer' : 300)
Page 3 : +150 offres (total'data engineer' : 450)
Page 4 : +150 offres (total'data engineer' : 600)
Page 5 : +107 offres (total'data engineer' : 707)
Dernière page atteinte
 Total pour 'data engineer': 707 offres

Récupération : 'data scientist'
Page 1 : +150 offres (total'data scientist' : 150)
Page 2 : +52 offres (total'data scientist' : 202)
Dernière page atteinte
 Total pour 'data scientist': 202 offres

Récupération : 'machine learning'
Page 1 : +150 offres (total'machine learning' : 150)
Page 2 : +150 offres (total'machine learning' : 300)
Page 3 : +150 offres (total'mach

In [7]:
!pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [8]:
import pandas as pd
import re

def extraire_salaire(libelle):
    if not libelle:
        return None, None
    nombres = re.findall(r'\d+(?:\.\d+)?', libelle)
    nombres = [float(n) for n in nombres if float(n) != 12.0]
    if len(nombres) == 0:
        return None, None
    elif len(nombres) == 1:
        return nombres[0], nombres[0]
    else:
        return nombres[0], nombres[1]

def flatten(o):
    #Saliaire
    salaire = o.get("salaire", {})
    salaire_lib = salaire.get("libelle")
    salaire_min, salaire_max = extraire_salaire(salaire_lib)

    #Lieu de travail
    lieu = o.get("lieuTravail", {})
    ville = lieu.get("libelle")
    code_postal = lieu.get("codePostal")
    commune = lieu.get("commune")

    #Entreprise
    entreprise = o.get("entreprise", {})
    nom_entreprise = entreprise.get("nom")
    secteur_entreprise = entreprise.get("seceteurActiviteLibelle")

    #Competence
    competences = o.get("competences", [])
    skills = ", ".join([c.get("libelle", "") for c in competences])

    #Formation requise
    formations = o.get("formations", [])
    niveaux_formation = ", ".join([f.get("niveauLibelle", "") for f in formations])

    #Langues requise
    langues = o.get("langues", [])
    langues_str = ", ".join([l.get("libelle", "") for l in langues])


    return {
        "id": o.get("id"),
        "intitule": o.get("intitule"),
        "description": o.get("description"),
        "type_contrat": o.get("typeContratLibelle"),
        "nb_heure_travail": o.get("dureeTravailLibelle"),
        "experience": o.get("experience"),
        "experience_exige": o.get("expereinceExige"),
        "salaire_libelle": salaire_lib,
        "salaire_min" : salaire_min,
        "salaire_max": salaire_max,
        "ville": ville,
        "code_postal": code_postal,
        "commune": commune,
        "entreprise": nom_entreprise,
        "secteur_activite": secteur_entreprise,
        "competences": competences,
        "formation_requise": niveaux_formation,
        "langues": langues,
        'type_de_poste': o.get("_type_de_poste"),
        "source": "France travail" 

    }

#Application de la fonction sur toute les offres
offres_extraites = [flatten(o) for o in toutes_offres]
df = pd.DataFrame(offres_extraites)
df = df.drop_duplicates(subset = "id").reset_index(drop = True)
print(f"Shape : {df.shape[0]} offres et {df.shape[1]} colonnes")
df.head()

Shape : 1634 offres et 20 colonnes


,id,intitule,description,type_contrat,nb_heure_travail,experience,experience_exige,salaire_libelle,salaire_min,salaire_max,ville,code_postal,commune,entreprise,secteur_activite,competences,formation_requise,langues,type_de_poste,source
0,206XPWH,Data Analyst H/F,Visaudio recherche pour sa Direction Analyse e...,CDI,39H/semaine\nTravail en journée,None,None,NaN,NaN,NaN,75 - PARIS 15,75015,75115,ECOUTER VOIR,None,"[{'code': '109527', 'libelle': 'Adapter les ou...",Bac+5 et plus ou équivalents,[],data analyst,France travail
1,206VKWR,Superviseur Business Intelligence - Data Analy...,Description du poste\n\nRattaché à notre branc...,CDI,35H/semaine\nTravail en journée,None,None,NaN,NaN,NaN,92 - Puteaux,92800,92062,RYDGE CONSEIL,None,"[{'code': '113255', 'libelle': 'Business Intel...",,[],data analyst,France travail
2,206TLCN,Data Analyst / Data Scientist Junior (H/F) B.E...,Dans le cadre du développement de nos activité...,CDI,35H/semaine\nTravail en journée...,None,None,Mensuel de 2150.0 Euros sur 12.0 mois,2150.0,2150.0,65 - Tarbes,65000,65440,BE.BELLANGER,None,"[{'code': '109527', 'libelle': 'Adapter les ou...",Bac+5 et plus ou équivalents,"[{'libelle': 'Anglais', 'exigence': 'S'}, {'li...",data analyst,France travail
3,206SSNV,Un(e) Data Analyst (H/F),La Communauté d'agglomération du bassin de Bri...,CDD - 12 Mois,35H/semaine\nTravail en journée,None,None,Mensuel de 1900.0 Euros à 1950.0 Euros sur 12....,1900.0,1950.0,19 - BRIVE-LA-GAILLARDE,19100,19031,COMMUNAUTE D'AGGLOMERATION DU BASSIN DE,None,"[{'code': '300067', 'libelle': 'Analyser, expl...",,[],data analyst,France travail
4,206SMJN,Superviseur Business Intelligence - Data Analy...,Description du poste\n\nRattaché à notre branc...,CDI,35H/semaine\nTravail en journée,None,None,NaN,NaN,NaN,92 - Puteaux,92800,92062,RYDGE CONSEIL,None,"[{'code': '113255', 'libelle': 'Business Intel...",,[],data analyst,France travail


Webscrapping de welcome to the jungle

In [9]:
pip install beautifulsoup4


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
def get_algolia_keys():
    r = requests.get(
        "https://www.welcometothejungle.com/fr/jobs",
        headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    )

    # Cherche les clés Algolia dans le HTML/JS de la page
    app_id  = re.search(r'algolia["\s:]+app[_\s]?id["\s:]+["\']([A-Z0-9]+)["\']', r.text, re.IGNORECASE)
    api_key = re.search(r'algolia["\s:]+api[_\s]?key["\s:]+["\']([a-z0-9]+)["\']', r.text, re.IGNORECASE)

    if app_id and api_key:
        print(f"APP_ID  : {app_id.group(1)}")
        print(f"API_KEY : {api_key.group(1)}")
        return app_id.group(1), api_key.group(1)
    else:
        # Clés connues en fallback (stables depuis 2023)
        print("Clés non trouvées dans le JS, utilisation des clés connues.")
        return "RQZL5DG1MF", "1c72d988a3abcfe01f5f83e4dfc40a68"

ALGOLIA_APP_ID, ALGOLIA_API_KEY = get_algolia_keys()

Clés non trouvées dans le JS, utilisation des clés connues.


In [11]:
from bs4 import BeautifulSoup
mots_cles = ["data analyst", "data engineer", "data scientist", "machine learning", "analyste données", "analyste quantitatif"]

ALGOLIA_URL = f"https://{ALGOLIA_APP_ID}-dsn.algolia.net/1/indexes/wttj_jobs_production_fr/query"

HEADERS_ALGOLIA = {
    "X-Algolia-Application-Id": ALGOLIA_APP_ID,
    "X-Algolia-API-Key":        ALGOLIA_API_KEY,
    "Content-Type":             "application/json",
}

OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok = True)

def fetch_page_wttj(mot_cle, page = 0, hits_per_page = 50) -> dict:
    url = "https://www.welcometothejungle.com/api/v1/jobs"
    payload = {
        "query": mot_cle,
        "page": page,
        "hitsPerPage": 50, 
        "filters": "offices.country_code:FR"
    }

    r = requests.post(ALGOLIA_URL, headers = HEADERS_ALGOLIA, json = payload)
    r.raise_for_status()
    return r.json()

def clean_html(texte):
    if not texte:
        return None
    return BeautifulSoup(texte, "html.parser").get_text(separator = " ").strip()

def flatten_wttj(o, mot_cle):
    #Salaire
    salaire = o.get("salary", {})
    salaire_min = salaire.get("min")
    salaire_max = salaire.get("max")
    salaire_devise = salaire.get("currency", "EUR")
    salaire_periode = salaire.get("period")

    #Construction du libelle du salaire
    if salaire_min and salaire_max:
        salaire_lib = f"{salaire_devise} {salaire_min} - {salaire_max}/{salaire_periode}"
    elif salaire_min:
        salaire_lib = f"{salaire_devise} {salaire_min}/{salaire_periode}"
    else:
        salaire_lib = None
    
    #Lieu de travail
    offices = o.get("offices", [{}])
    lieu = offices[0] if offices else {}
    ville = lieu.get("city")
    code_postal = lieu.get("zip_code")
    commune = lieu.get("district")

    #Entreprise
    entreprise = o.get("organization", {})
    nom_entreprise = entreprise.get("name")
    secteur_brut = o.get("sector")
    secteur_entreprise = secteur_brut.get("name") if secteur_brut else None

    #Contrat
    contrat_map = {
        "FULL_TIME" : "CDI",
        "PART_TIME": "Temps partiel",
        "INTERNSHIP" : "Stage",
        "FREELANCE" : "Freelance",
        "APPRENTICESHIP": "Alternance",
        "VIE": "VIE",
    }
    contrat_raw = o.get("contract_type")
    contrat = contrat_map.get(contrat_raw, contrat_raw)

    # Expérience
    exp_map = {
        "LESS_THAN_6_MONTHS":  "Moins de 6 mois",
        "6_MONTHS_TO_1_YEAR":  "6 mois à 1 an",
        "1_TO_2_YEARS":        "1 à 2 ans",
        "3_TO_5_YEARS":        "3 à 5 ans",
        "6_TO_10_YEARS":       "6 à 10 ans",
        "MORE_THAN_10_YEARS":  "Plus de 10 ans",
    }
    experience = exp_map.get(o.get("experience_level"), o.get("experience_level"))

    # Compétences (tags)
    tags       = o.get("tags", []) or []
    competences = ", ".join([t.get("name", "") for t in tags])

    #Description
    description = clean_html(o.get("description"))
    profil = clean_html(o.get("profile"))
    description_complete = f"{description or ''} {profil or''}".strip()


    #Formation requise
    edu_map = {
        "BAC": "Bac",
        "BAC_2": "Bac+2",
        "BAC_3": "Bac+3",
        "BAC_4": "Bac+4",
        "BAC_5": "Bac+5",
        "PHD": "Doctorat"
    }
    formations = edu_map.get(o.get("education_level"), o.get("education_level"))


    return {
        "id": o.get("reference"),
        "intitule": o.get("name"),
        "description": description_complete,
        "type_contrat": contrat,
        "nb_heure_travail": None,
        "experience": experience,
        "experience_exige": o.get("experience_level"),
        "salaire_libelle": salaire_lib,
        "salaire_min" : float(salaire_min) if salaire_min else None,
        "salaire_max": float(salaire_max) if salaire_max else None,
        "ville": ville,
        "code_postal": code_postal,
        "commune": commune,
        "entreprise": nom_entreprise,
        "secteur_activite": secteur_entreprise,
        "competences": competences,
        "formation_requise": formations,
        "langues": None,
        'type_de_poste': mot_cle,
        "source": "welcometothejungle"

    }

toutes_offres_wttj = []

for mot_cle in mots_cles :
    print(f"\nRécupération : '{mot_cle}'")
    offres_recherchees = []
    page = 0
    nb_pages = 1

    while page < nb_pages:
        try: 
            result = fetch_page_wttj(mot_cle, page = page)
            hits = result.get("hits", [])
            nb_pages = result.get("nbPages", 1)
            total = result.get("nbHits", "?")

            if not hits:
                print(f"Page {page}: vide, arrêt.")
                break

            for o in hits :
                toutes_offres_wttj.append(flatten_wttj(o, mot_cle))
                o["_type_de_poste"] = mot_cle

            print(f"Page {page+1}/{nb_pages} : {len(hits)} offres (total '{mot_cle}' : {total})")
           
            page += 1
            time.sleep(0.5)

        except requests.HTTPError as e:
            print(f" Erreur HTTP page {page} : {e}")
            break
        

print(f"\n{'='*45}")
print(f"Total Général: {len(toutes_offres_wttj)} offres collectées")
print(f"{'='*45}")

#Application et construction du dataframe

offres_extraites_wttj = [flatten_wttj(o, o["_type_de_poste"]) for o in toutes_offres_wttj]
df_wttj = pd.DataFrame(offres_extraites_wttj)
df_wttj = df_wttj.drop_duplicates(subset = "id").reset_index(drop = True)
print(f"Shape : {df_wttj.shape[0]} offres et {df_wttj.shape[1]} colonnes")
df_wttj.head()


Récupération : 'data analyst'


ConnectionError: HTTPSConnectionPool(host='rqzl5dg1mf-dsn.algolia.net', port=443): Max retries exceeded with url: /1/indexes/wttj_jobs_production_fr/query (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7bd16bb34b90>: Failed to resolve 'rqzl5dg1mf-dsn.algolia.net' ([Errno -2] Name or service not known)"))

In [12]:
print(f"APP_ID  : '{ALGOLIA_APP_ID}'")
print(f"API_KEY : '{ALGOLIA_API_KEY}'")
print(f"URL construite : {ALGOLIA_URL}")

APP_ID  : 'RQZL5DG1MF'
API_KEY : '1c72d988a3abcfe01f5f83e4dfc40a68'
URL construite : https://RQZL5DG1MF-dsn.algolia.net/1/indexes/wttj_jobs_production_fr/query
